<a href="https://colab.research.google.com/github/erpanter/AdvancedTopicsInAI/blob/main/week13_Transformer/bert-transfer-learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment Transfer Learning with BERT for Text Classification

One of the approaches where we can use BERT for downstream task such as text classification is to do fine-tuning of the pretrained model.

In this lab, we will see how we can use a pretrained DistilBert Model and fine-tune it with custom training data for text classification task.

At the end of this session, you will be able to:

prepare data and use model-specific Tokenizer to format data suitable for use by the model
configure the transformer model for fine-tuning
train the model for binary and multi-class text classification

## Install Transformers and other libraries

If you are running this notebook in Google Colab, you will need to install the Hugging Face transformers library as it is not part of the standard environment.


In [1]:
%%capture
!pip install datasets>=2.18.0 transformers>=4.38.2 sentence-transformers>=2.5.1 setfit>=1.0.3 accelerate>=0.27.2 seqeval>=1.2.2

## Prepare Dataset

The train set has 40000 samples. We will use only a small subset (e.g. 2500) samples for finetuning our pretrained model. Similarly we will use a smaller test set for evaluating our model.

In [2]:
import numpy as np
from datasets import load_dataset

# downloaded the datasets.
test_data_url = 'https://nyp-aicourse.s3-ap-southeast-1.amazonaws.com/datasets/imdb_test.csv'
train_data_url = 'https://nyp-aicourse.s3-ap-southeast-1.amazonaws.com/datasets/imdb_train.csv'

train_data = load_dataset('csv', data_files=train_data_url, split="train").shuffle(seed=128).select(range(2500))
test_data = load_dataset('csv', data_files=test_data_url, split="train").shuffle(seed=128).select(range(500))

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load Model and Tokenizer
model_id = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_id)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

### Tokenization

Let us take a closer look at the output of the tokenization process.

We notice that the tokenizer will return a dictionary of two items 'input_ids' and 'attention_mask'. The input_ids contains the IDs of the tokens. While the 'attention_mask' contains the masking pattern for those padded positions. If you are using BERT tokenizer, there will be additional item called 'token_type_ids'.

We also notice that for the example sentence, the word 'Transformer' is being broken up into two tokens 'Trans' and '##former'. The '##' means that the rest of the token should be attached to the previous one.

We also see that the tokenizer appended [CLS] (token_id=101) to the beginning of the token sequence, and [SEP] (token_id=102) at the end.

In [4]:
test_sentence = "Transformer is really good for Natural Language Processing."

encoding = tokenizer(test_sentence, padding=True, truncation=True)
print(f"Encoding keys:  {encoding.keys()}\n")

print(f"token ids: {encoding['input_ids']}\n")
print(f"attention_mask: {encoding['attention_mask']}\n")
print(f"tokens: {tokenizer.decode(encoding['input_ids'])}")

Encoding keys:  KeysView({'input_ids': [101, 10938, 2121, 2003, 2428, 2204, 2005, 3019, 2653, 6364, 1012, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]})

token ids: [101, 10938, 2121, 2003, 2428, 2204, 2005, 3019, 2653, 6364, 1012, 102]

attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

tokens: [CLS] transformer is really good for natural language processing. [SEP]


###Create the tokenized dataset

We will first convert the sentiment label from text to numeric label. We will need to create a data field called 'label' as the model will use this 'label' key as the target label.

We will also use the model's (in this case the DistilBERT) tokenizer to produce the input data that are suitable to be used by the DistilBert model, e.g. the input_ids, the attention_mask.  It automatically append the [CLS] token in the front of the sequence of token_ids and the [SEP] token at the end of the sequence of token_ids , and also the attention mask for those padded positions in the input sequence of tokens.

We also specify the DataCollator to use. Data collators are objects that will form a batch by using a list of dataset elements as input. These elements are of the same type as the elements of train_dataset or eval_dataset. To be able to build batches, data collators may apply some processing (like padding).

In [5]:
from transformers import DataCollatorWithPadding

# Pad to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def convert_label(sample):
    return {
        "label" : 0 if sample['sentiment'] == 'negative' else 1
    }
    # sample['sentiment'] = 0 if sample['sentiment'] == 'negative' else 1
    # return { "label": sample['sentiment']}
    # return sample
    # return {
    #     "text":  sample['review'],
    #     "label": sample['sentiment'] }

def preprocess_function(examples):
    """Tokenize input data"""
    return tokenizer(examples["review"], truncation=True)


# Tokenize train/test data
tokenized_train = train_data.map(convert_label).map(preprocess_function, remove_columns=['review', 'sentiment'], batched=True)
tokenized_test = test_data.map(convert_label).map(preprocess_function, remove_columns=['review', 'sentiment'], batched=True)

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [6]:
print(tokenized_test)

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 500
})


We will define a compute_metrics() function to calculate the necessary metrics. With compute_metrics we can define any number of metrics that we are
interested in and that can be printed out or logged during training. This is
especially helpful during training as it allows for detecting overfitting
behavior.

In [7]:
import numpy as np
import evaluate


def compute_metrics(eval_pred):
    """Calculate F1 score"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    load_f1 = evaluate.load("f1")
    f1 = load_f1.compute(predictions=predictions, references=labels)["f1"]
    return {"f1": f1}

## Train the Model

We will instantiate a pretrained model 'distilbert-base-uncased', using AutoModelForSequenceClassification.

We define the number of labels that we want to predict beforehand. This is
necessary to create the feedforward neural network that is applied on top of
our pretrained model:

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Transformer models benefit from a much lower learning rate than the default used by AdamW, which is 0.001. In this training, we will start the training with 2e-5 (0.00002) and slowly reduce the learning rate over the course of training. In the literature, you will sometimes see this referred to as decaying or annealing the learning rate.

In [9]:
import os
import wandb

# go to https://wandb.ai/authorize to get your access key
wandb.login(key="cfbc55fe9459fffe1dfc511249cc81bf987b4010")
os.environ['WANDB_PROJECT']="transformer_proj"

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nanyingliang (is2b) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [10]:
from transformers import TrainingArguments, Trainer

# Training arguments for parameter tuning
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   eval_strategy='steps',
   eval_steps=0.1,
   report_to="wandb",
   logging_steps=0.1,
   run_name="bert-finetune"
)

# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)


/tmp/ipython-input-10-770826306.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [11]:
trainer.train()

Step,Training Loss,Validation Loss,F1
16,0.696200,0.679661,0.228188
32,0.674400,0.641916,0.766595
48,0.602000,0.526721,0.763218
64,0.467400,0.381296,0.871094
80,0.366300,0.328454,0.880000
96,0.335400,0.306665,0.879845
112,0.326400,0.313726,0.862986
128,0.325500,0.299345,0.882236
144,0.272500,0.309279,0.873469


TrainOutput(global_step=157, training_loss=0.436784633405649, metrics={'train_runtime': 193.9659, 'train_samples_per_second': 12.889, 'train_steps_per_second': 0.809, 'total_flos': 326999913188544.0, 'train_loss': 0.436784633405649, 'epoch': 1.0})

Evaluate results.

In [12]:
trainer.evaluate()

{'eval_loss': 0.3123502731323242,
 'eval_f1': 0.865979381443299,
 'eval_runtime': 7.6018,
 'eval_samples_per_second': 65.774,
 'eval_steps_per_second': 4.21,
 'epoch': 1.0}

### Freeze Layers

To show the importance of training the entire network, we will now freeze the main DistilBERT model and allow only updates to pass through the classification head.

In [13]:
# Load Model and Tokenizer
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Our pretrained DistilBERT model contains a lot of layers that we can potentially
freeze. Inspecting these layers gives insight into the structure of the network
and what we might want to freeze:

In [14]:
# Print layer names
for name, param in model.named_parameters():
    print(name)

distilbert.embeddings.word_embeddings.weight
distilbert.embeddings.position_embeddings.weight
distilbert.embeddings.LayerNorm.weight
distilbert.embeddings.LayerNorm.bias
distilbert.transformer.layer.0.attention.q_lin.weight
distilbert.transformer.layer.0.attention.q_lin.bias
distilbert.transformer.layer.0.attention.k_lin.weight
distilbert.transformer.layer.0.attention.k_lin.bias
distilbert.transformer.layer.0.attention.v_lin.weight
distilbert.transformer.layer.0.attention.v_lin.bias
distilbert.transformer.layer.0.attention.out_lin.weight
distilbert.transformer.layer.0.attention.out_lin.bias
distilbert.transformer.layer.0.sa_layer_norm.weight
distilbert.transformer.layer.0.sa_layer_norm.bias
distilbert.transformer.layer.0.ffn.lin1.weight
distilbert.transformer.layer.0.ffn.lin1.bias
distilbert.transformer.layer.0.ffn.lin2.weight
distilbert.transformer.layer.0.ffn.lin2.bias
distilbert.transformer.layer.0.output_layer_norm.weight
distilbert.transformer.layer.0.output_layer_norm.bias
distil

In [15]:
for name, param in model.named_parameters():

     # Trainable classification head
     if name.startswith("classifier") or name.startswith("pre_classifier"):
        param.requires_grad = True

      # Freeze everything else
     else:
        param.requires_grad = False

In [16]:
# We can check whether the model was correctly updated
for name, param in model.named_parameters():
     print(f"Parameter: {name} ----- {param.requires_grad}")

Parameter: distilbert.embeddings.word_embeddings.weight ----- False
Parameter: distilbert.embeddings.position_embeddings.weight ----- False
Parameter: distilbert.embeddings.LayerNorm.weight ----- False
Parameter: distilbert.embeddings.LayerNorm.bias ----- False
Parameter: distilbert.transformer.layer.0.attention.q_lin.weight ----- False
Parameter: distilbert.transformer.layer.0.attention.q_lin.bias ----- False
Parameter: distilbert.transformer.layer.0.attention.k_lin.weight ----- False
Parameter: distilbert.transformer.layer.0.attention.k_lin.bias ----- False
Parameter: distilbert.transformer.layer.0.attention.v_lin.weight ----- False
Parameter: distilbert.transformer.layer.0.attention.v_lin.bias ----- False
Parameter: distilbert.transformer.layer.0.attention.out_lin.weight ----- False
Parameter: distilbert.transformer.layer.0.attention.out_lin.bias ----- False
Parameter: distilbert.transformer.layer.0.sa_layer_norm.weight ----- False
Parameter: distilbert.transformer.layer.0.sa_layer_

In [17]:
from transformers import TrainingArguments, Trainer

# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)
trainer.train()

/tmp/ipython-input-17-2563977997.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,F1
16,0.697900,0.695863,0.276215
32,0.696700,0.691487,0.678621
48,0.687600,0.688278,0.693098
64,0.688400,0.685679,0.702290
80,0.689200,0.684088,0.482574
96,0.685500,0.681726,0.707368
112,0.682600,0.680592,0.663636
128,0.680600,0.679847,0.609337
144,0.684400,0.679395,0.564103


TrainOutput(global_step=157, training_loss=0.6875311857575823, metrics={'train_runtime': 108.6842, 'train_samples_per_second': 23.002, 'train_steps_per_second': 1.445, 'total_flos': 326999913188544.0, 'train_loss': 0.6875311857575823, 'epoch': 1.0})

In [18]:
trainer.evaluate()

{'eval_loss': 0.6792719960212708,
 'eval_f1': 0.5641025641025641,
 'eval_runtime': 7.642,
 'eval_samples_per_second': 65.428,
 'eval_steps_per_second': 4.187,
 'epoch': 1.0}

### Freeze blocks 1-4

Instead of freezing nearly all layers, let’s freeze everything up until encoder block 4 and see how it affects performance. A major benefit is that this reduces computation but still allows updates to flow through part of the
pretrained model:

In [19]:
# We can check whether the model was correctly updated
for index, (name, param) in enumerate(model.named_parameters()):
     print(f"Parameter: {index}{name} ----- {param.requires_grad}")

Parameter: 0distilbert.embeddings.word_embeddings.weight ----- False
Parameter: 1distilbert.embeddings.position_embeddings.weight ----- False
Parameter: 2distilbert.embeddings.LayerNorm.weight ----- False
Parameter: 3distilbert.embeddings.LayerNorm.bias ----- False
Parameter: 4distilbert.transformer.layer.0.attention.q_lin.weight ----- False
Parameter: 5distilbert.transformer.layer.0.attention.q_lin.bias ----- False
Parameter: 6distilbert.transformer.layer.0.attention.k_lin.weight ----- False
Parameter: 7distilbert.transformer.layer.0.attention.k_lin.bias ----- False
Parameter: 8distilbert.transformer.layer.0.attention.v_lin.weight ----- False
Parameter: 9distilbert.transformer.layer.0.attention.v_lin.bias ----- False
Parameter: 10distilbert.transformer.layer.0.attention.out_lin.weight ----- False
Parameter: 11distilbert.transformer.layer.0.attention.out_lin.bias ----- False
Parameter: 12distilbert.transformer.layer.0.sa_layer_norm.weight ----- False
Parameter: 13distilbert.transformer

In [20]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Encoder block 10 starts at index 68 and
# we freeze everything before that block
for index, (name, param) in enumerate(model.named_parameters()):
    if index < 68:
        param.requires_grad = False



Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

/tmp/ipython-input-21-4230016659.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,F1
16,0.696500,0.688152,0.094203
32,0.687300,0.675228,0.719424
48,0.665000,0.653901,0.700000
64,0.641400,0.611870,0.742729
80,0.596200,0.549830,0.818182
96,0.523700,0.482535,0.851224
112,0.455700,0.452097,0.797357
128,0.432000,0.411157,0.836820
144,0.406700,0.390422,0.842105


{'eval_loss': 0.3879224359989166,
 'eval_f1': 0.8391038696537678,
 'eval_runtime': 7.6479,
 'eval_samples_per_second': 65.378,
 'eval_steps_per_second': 4.184,
 'epoch': 1.0}

Fine-tune vs transfer learning
which method is better?